# 🌑 LunarSight — Notebook 05: Pathfinding

**Agent 5**: Plan a physically survivable rover traverse from crater rim
to ice deposit using kinodynamic A* with terramechanics constraints.

---

In [ ]:
# === Setup ===
import os
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/Lunar-Sight'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/YOUR_USERNAME/Lunar-Sight.git {REPO_DIR}
os.chdir(os.path.join(REPO_DIR, 'Lunar-Sight'))
!pip install -q -r requirements_colab.txt

In [ ]:
# === Run Agent 5 ===
import yaml, logging
logging.basicConfig(level=logging.INFO)

from agent5_pathfinding.agent import agent5_node

state = {
    'mission_config_path': 'config/mission_config.yaml',
    'ice_mask_path': 'outputs/agent4/ice_mask.npy',
    'confidence_map_path': 'outputs/agent4/confidence_map.npy',
    'slope_path': 'outputs/agent1/slope.npy',
    'dem_path': 'outputs/agent1/dem.npy',
    'pixel_size_m': 118.0,
}

result = agent5_node(state)
print(f"Status: {result.get('agent5_status')}")
print(f"Distance: {result.get('path_distance_m', 0):.1f} m")
print(f"Energy: {result.get('path_energy_wh', 0):.2f} Wh")

In [ ]:
# === Visualize Path ===
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

slope = np.load('outputs/agent1/slope.npy')
ice_mask = np.load('outputs/agent4/ice_mask.npy')
path = result.get('traverse_path', [])

if path:
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    
    # Path over slope map
    im0 = axes[0].imshow(slope, cmap='RdYlGn_r', vmin=0, vmax=20)
    path_y = [wp['y'] for wp in path]
    path_x = [wp['x'] for wp in path]
    axes[0].plot(path_x, path_y, 'b-', linewidth=2, label='Traverse')
    axes[0].plot(path_x[0], path_y[0], 'g^', markersize=12, label='Start')
    axes[0].plot(path_x[-1], path_y[-1], 'r*', markersize=15, label='Ice Target')
    axes[0].legend()
    axes[0].set_title('Planned Traverse over Slope Map')
    plt.colorbar(im0, ax=axes[0], label='Slope (°)')
    
    # Battery profile
    battery = [wp['battery_wh'] for wp in path]
    axes[1].plot(battery, 'g-', linewidth=2)
    axes[1].fill_between(range(len(battery)), battery, alpha=0.3, color='green')
    axes[1].set_xlabel('Waypoint')
    axes[1].set_ylabel('Battery (Wh)')
    axes[1].set_title('Battery Depletion Profile')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print('No path found — try adjusting constraints')